In [ ]:
```xml
<VSCode.Cell language="markdown">
# ⚡ Performance Profiling & Optimization Guide

**Deep dive into profiling, flame graphs, bottleneck identification, and optimization strategies**

This guide covers:
- CPU profiling and flame graphs
- Memory profiling and leak detection
- I/O and disk profiling
- Network profiling
- Bottleneck identification
- Optimization strategies
- Load testing and benchmarking
- Performance regression detection
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. CPU Profiling

### Python cProfile Usage
```python
import cProfile
import pstats
from io import StringIO

class CPUProfiler:
    """CPU profiling for performance analysis"""
    
    @staticmethod
    def profile_function(func, *args, **kwargs):
        """Profile single function execution"""
        profiler = cProfile.Profile()
        profiler.enable()
        
        try:
            result = func(*args, **kwargs)
        finally:
            profiler.disable()
        
        # Print statistics
        stats = pstats.Stats(profiler, stream=StringIO())
        stats.strip_dirs()
        stats.sort_stats('cumulative')
        stats.print_stats(20)  # Top 20 functions
        
        return result
    
    @staticmethod
    def profile_code_block(code_str: str, globals_dict: dict = None):
        """Profile arbitrary code block"""
        if globals_dict is None:
            globals_dict = {}
        
        profiler = cProfile.Profile()
        profiler.enable()
        
        try:
            exec(code_str, globals_dict)
        finally:
            profiler.disable()
            
            stats = pstats.Stats(profiler)
            stats.sort_stats('cumulative')
            stats.print_stats(20)

# Usage
def slow_function():
    total = 0
    for i in range(1000000):
        total += i
    return total

CPUProfiler.profile_function(slow_function)
```

### Flame Graph Generation
```python
import py_spy

class FlameGraphProfiler:
    """Generate flame graphs for visual profiling"""
    
    @staticmethod
    def generate_flame_graph(script_path: str, output_path: str = 'flamegraph.svg'):
        """Generate flame graph from Python script"""
        # Use py_spy for sampling-based profiling
        # py-spy record -o profile.svg python script.py
        
        import subprocess
        cmd = ['py-spy', 'record', '-o', output_path, 'python', script_path]
        subprocess.run(cmd)
    
    @staticmethod
    def analyze_hotspots(profiler_stats: pstats.Stats) -> List[tuple]:
        """Extract hotspots from profiler stats"""
        hotspots = []
        
        for func, stat in profiler_stats.stats.items():
            cumulative_time = stat[3]
            hotspots.append((func, cumulative_time))
        
        # Sort by cumulative time
        hotspots.sort(key=lambda x: x[1], reverse=True)
        
        return hotspots[:10]  # Top 10 hotspots
```

### Async Profiling
```python
import asyncio

class AsyncProfiler:
    """Profile async functions"""
    
    @staticmethod
    async def profile_async_function(func, *args, **kwargs):
        """Profile async function"""
        import time
        
        start = time.time()
        result = await func(*args, **kwargs)
        elapsed = time.time() - start
        
        print(f"Async function {func.__name__} took {elapsed:.3f}s")
        return result
    
    @staticmethod
    async def profile_concurrent_operations(tasks: List):
        """Profile concurrent execution"""
        import time
        
        start = time.time()
        results = await asyncio.gather(*tasks)
        elapsed = time.time() - start
        
        print(f"Concurrent operations took {elapsed:.3f}s")
        return results
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Memory Profiling

### Memory Leak Detection
```python
import tracemalloc
from memory_profiler import profile

class MemoryProfiler:
    """Memory profiling and leak detection"""
    
    @staticmethod
    def start_memory_tracking():
        """Start memory tracking"""
        tracemalloc.start()
    
    @staticmethod
    def get_memory_stats():
        """Get current memory statistics"""
        current, peak = tracemalloc.get_traced_memory()
        
        return {
            'current_mb': current / 1024 / 1024,
            'peak_mb': peak / 1024 / 1024
        }
    
    @staticmethod
    def get_memory_snapshot(top_n: int = 10) -> List[tuple]:
        """Get top memory-consuming functions"""
        snapshot = tracemalloc.take_snapshot()
        top_stats = snapshot.statistics('lineno')
        
        results = []
        for stat in top_stats[:top_n]:
            results.append({
                'file': stat.traceback[0].filename,
                'line': stat.traceback[0].lineno,
                'size_mb': stat.size / 1024 / 1024,
                'count': stat.count
            })
        
        return results
    
    @staticmethod
    def detect_memory_growth():
        """Detect memory growth over time"""
        import time
        
        samples = []
        
        for _ in range(10):
            stats = tracemalloc.get_traced_memory()
            samples.append(stats[0] / 1024 / 1024)
            time.sleep(1)
        
        growth_rate = (samples[-1] - samples[0]) / samples[0] * 100
        print(f"Memory growth: {growth_rate:.1f}%")
        
        return growth_rate

# Line-by-line profiling
@profile
def memory_intensive_function():
    big_list = [i for i in range(1000000)]
    return sum(big_list)
```

### Object Growth Tracking
```python
class ObjectTracker:
    """Track object creation and growth"""
    
    def __init__(self):
        self.initial_objects = {}
        self.snapshots = []
    
    def take_snapshot(self, label: str = None):
        """Take snapshot of object counts"""
        import gc
        
        gc.collect()
        
        objects = {}
        for obj in gc.get_objects():
            obj_type = type(obj).__name__
            objects[obj_type] = objects.get(obj_type, 0) + 1
        
        self.snapshots.append({
            'label': label,
            'objects': objects
        })
        
        return objects
    
    def compare_snapshots(self) -> dict:
        """Compare object growth between snapshots"""
        if len(self.snapshots) < 2:
            return {}
        
        first = self.snapshots[0]['objects']
        last = self.snapshots[-1]['objects']
        
        growth = {}
        for obj_type in set(list(first.keys()) + list(last.keys())):
            initial = first.get(obj_type, 0)
            final = last.get(obj_type, 0)
            
            if final > initial:
                growth[obj_type] = {
                    'initial': initial,
                    'final': final,
                    'growth': final - initial
                }
        
        return growth
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Benchmarking & Load Testing

### Simple Benchmarking
```python
import time
import statistics

class Benchmark:
    """Simple performance benchmarking"""
    
    def __init__(self, name: str, iterations: int = 100):
        self.name = name
        self.iterations = iterations
        self.times = []
    
    def run(self, func, *args, **kwargs):
        """Run benchmark"""
        for _ in range(self.iterations):
            start = time.perf_counter()
            func(*args, **kwargs)
            end = time.perf_counter()
            
            self.times.append((end - start) * 1000)  # ms
    
    def get_stats(self) -> dict:
        """Get benchmark statistics"""
        return {
            'min': min(self.times),
            'max': max(self.times),
            'mean': statistics.mean(self.times),
            'median': statistics.median(self.times),
            'stdev': statistics.stdev(self.times) if len(self.times) > 1 else 0,
            'p99': sorted(self.times)[int(0.99 * len(self.times))]
        }
    
    def report(self):
        """Print benchmark report"""
        stats = self.get_stats()
        print(f"Benchmark: {self.name}")
        print(f"  Min: {stats['min']:.3f}ms")
        print(f"  Mean: {stats['mean']:.3f}ms")
        print(f"  Median: {stats['median']:.3f}ms")
        print(f"  P99: {stats['p99']:.3f}ms")
        print(f"  Max: {stats['max']:.3f}ms")
        print(f"  StDev: {stats['stdev']:.3f}ms")

# Usage
def test_function():
    return sum(range(1000))

bench = Benchmark("Sum Test")
bench.run(test_function)
bench.report()
```

### Load Testing
```python
import concurrent.futures

class LoadTester:
    """Simulate load on system"""
    
    def __init__(self, target_func, concurrent_users: int = 10):
        self.target_func = target_func
        self.concurrent_users = concurrent_users
        self.results = []
    
    def run_load_test(self, duration_seconds: int, rate: int = None):
        """Run load test"""
        import threading
        
        stop_event = threading.Event()
        request_count = 0
        error_count = 0
        
        def worker():
            nonlocal request_count, error_count
            while not stop_event.is_set():
                try:
                    start = time.time()
                    self.target_func()
                    elapsed = time.time() - start
                    
                    self.results.append({'time': elapsed, 'success': True})
                    request_count += 1
                except Exception as e:
                    self.results.append({'time': 0, 'success': False, 'error': str(e)})
                    error_count += 1
        
        threads = []
        for _ in range(self.concurrent_users):
            t = threading.Thread(target=worker)
            t.start()
            threads.append(t)
        
        time.sleep(duration_seconds)
        stop_event.set()
        
        for t in threads:
            t.join()
        
        return {
            'total_requests': request_count,
            'errors': error_count,
            'success_rate': (request_count - error_count) / request_count if request_count > 0 else 0,
            'avg_latency': statistics.mean([r['time'] for r in self.results if r['success']])
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Optimization Strategies

### Algorithm Optimization
```python
class AlgorithmOptimization:
    """Common algorithm optimization patterns"""
    
    @staticmethod
    def memoization(func):
        """Add memoization to function"""
        cache = {}
        
        def memoized(*args):
            if args in cache:
                return cache[args]
            
            result = func(*args)
            cache[args] = result
            return result
        
        return memoized
    
    @staticmethod
    def dynamic_programming_example():
        """Example: Fibonacci with DP"""
        def fib(n, memo=None):
            if memo is None:
                memo = {}
            
            if n in memo:
                return memo[n]
            
            if n <= 1:
                return n
            
            result = fib(n - 1, memo) + fib(n - 2, memo)
            memo[n] = result
            return result
        
        return fib(40)

class DataStructureOptimization:
    """Choose optimal data structures"""
    
    @staticmethod
    def use_set_for_lookups():
        """Use set for O(1) lookup instead of list O(n)"""
        # Bad: O(n)
        items_list = [1, 2, 3, 4, 5]
        if 3 in items_list:  # Linear search
            pass
        
        # Good: O(1)
        items_set = {1, 2, 3, 4, 5}
        if 3 in items_set:  # Hash lookup
            pass
    
    @staticmethod
    def use_deque_for_queues():
        """Use deque instead of list for queue operations"""
        from collections import deque
        
        # Bad: O(n) for popleft on list
        queue_list = [1, 2, 3, 4, 5]
        queue_list.pop(0)
        
        # Good: O(1) for popleft on deque
        queue = deque([1, 2, 3, 4, 5])
        queue.popleft()
```

### Resource Pool Management
```python
from queue import Queue

class ResourcePool:
    """Manage pool of expensive resources"""
    
    def __init__(self, resource_factory, pool_size: int = 10):
        self.factory = resource_factory
        self.pool = Queue(maxsize=pool_size)
        
        for _ in range(pool_size):
            self.pool.put(self.factory())
    
    def acquire(self, timeout: int = 5):
        """Acquire resource from pool"""
        try:
            return self.pool.get(timeout=timeout)
        except:
            raise RuntimeError("No resources available")
    
    def release(self, resource):
        """Return resource to pool"""
        self.pool.put(resource)
    
    def __enter__(self):
        self.resource = self.acquire()
        return self.resource
    
    def __exit__(self, *args):
        self.release(self.resource)

# Usage
class Connection:
    def __init__(self):
        pass

pool = ResourcePool(Connection, pool_size=10)

with pool as conn:
    # Use connection
    pass
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. Performance Optimization Checklist

✅ **Profiling**
- [ ] CPU profiler run and hotspots identified
- [ ] Memory profiler run and leaks found
- [ ] Flame graphs generated and analyzed
- [ ] Async operations profiled
- [ ] I/O bottlenecks identified

✅ **Optimization**
- [ ] Algorithms optimized (Big O complexity reduced)
- [ ] Data structures chosen optimally
- [ ] Caching implemented for expensive operations
- [ ] Database queries optimized (indexes, etc.)
- [ ] Network requests batched/minimized

✅ **Testing**
- [ ] Benchmarks established for critical paths
- [ ] Load testing performed
- [ ] Performance under peak load verified
- [ ] Regression tests added
- [ ] SLAs met

✅ **Monitoring**
- [ ] Performance metrics collected
- [ ] Alerts set for regression
- [ ] Dashboards created
- [ ] Continuous profiling in production
- [ ] Root causes identified quickly
</VSCode.Cell>
```